# Bootstrap: Grafana OTEL → Databricks secret

**Committed template (no secrets).** Copy to `secrets-bootstrap.local.ipynb` (gitignored) and paste real values there, **or** fill the variables in a private clone and never commit that edit.

Writes scope **`lfczerobusdemo`** / key **`otel-grafana-rslee6392`** — same JSON shape as `zerobus-otel.ipynb`.

Get values from Grafana → **Connections** → **OpenTelemetry (collector)** (see `zerobus-otel.ipynb` intro).

In [ ]:
%pip install --quiet databricks-sdk

In [ ]:
# Paste from Grafana (GRAFANA_CLOUD_* in the UI). Leave token/header empty until you have real values.

GRAFANA_OTLP_ENDPOINT = ""  # e.g. https://otlp-gateway-prod-us-west-0.grafana.net
GRAFANA_BASIC_AUTH_HEADER = ""  # full "Basic …" from GRAFANA_CLOUD_BASIC_AUTH_HEADER

# Optional if you skip BASIC_AUTH_HEADER: instance id + API token (notebook / zerobus-otel can build Basic)
GRAFANA_INSTANCE_ID = ""
GRAFANA_API_TOKEN = ""

OTEL_EXPORTER_TYPE = "grafana"

In [ ]:
import json

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists

_SECRET_SCOPE = "lfczerobusdemo"
_OTEL_GRAFANA_KEY = "otel-grafana-rslee6392"

payload = {
    "OTEL_EXPORTER_TYPE": OTEL_EXPORTER_TYPE,
    "GRAFANA_OTLP_ENDPOINT": GRAFANA_OTLP_ENDPOINT.strip(),
    "GRAFANA_BASIC_AUTH_HEADER": GRAFANA_BASIC_AUTH_HEADER.strip(),
    "GRAFANA_INSTANCE_ID": GRAFANA_INSTANCE_ID.strip(),
    "GRAFANA_API_TOKEN": GRAFANA_API_TOKEN.strip(),
}

if not payload["GRAFANA_BASIC_AUTH_HEADER"] and not (
    payload["GRAFANA_INSTANCE_ID"] and payload["GRAFANA_API_TOKEN"]
):
    raise ValueError(
        "Set GRAFANA_BASIC_AUTH_HEADER (recommended) or both GRAFANA_INSTANCE_ID and GRAFANA_API_TOKEN"
    )

w = WorkspaceClient()

try:
    w.secrets.create_scope(_SECRET_SCOPE)
    print(f"Created scope {_SECRET_SCOPE!r}")
except ResourceAlreadyExists:
    print(f"Scope {_SECRET_SCOPE!r} already exists")

current = {}
try:
    raw = dbutils.secrets.get(scope=_SECRET_SCOPE, key=_OTEL_GRAFANA_KEY)
    current = json.loads(raw)
except Exception:
    pass

current.update(payload)
w.secrets.put_secret(
    scope=_SECRET_SCOPE,
    key=_OTEL_GRAFANA_KEY,
    string_value=json.dumps(current, indent=2),
)
print(f"Updated {_SECRET_SCOPE}/{_OTEL_GRAFANA_KEY} keys: {list(payload.keys())}")